In [1]:
import pandas as pd, numpy as np, glob, os

def load_clean(path, benign_frac=1.0):
    d = pd.read_csv(path)
    d.columns = d.columns.str.strip()
    d = d.drop(columns=["Fwd Header Length.1"], errors="ignore")
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    num = d.select_dtypes("number").columns
    d[num] = d[num].astype("float32")
    name = os.path.basename(path)
    d["day"] = name.split("-")[0]
    d["file"] = name
    if benign_frac < 1.0:                       # optional downsampling to save memory
        keep = (d["Label"] != "BENIGN") | (np.random.rand(len(d)) < benign_frac)
        d = d[keep]
    return d

files = sorted(glob.glob("../data/raw/*.csv"))
for f in files: print(os.path.basename(f))

df = pd.concat([load_clean(f) for f in files], ignore_index=True)
print("before dedup:", df.shape)
df = df.drop_duplicates(subset=[c for c in df.columns if c not in ["day", "file"]])
print("after dedup:", df.shape)
print(df.groupby("day")["Label"].value_counts())

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv
before dedup: (2827876, 80)
after dedup: (2497980, 80)
day        Label                     
Friday     BENIGN                        394764
           DDoS                          128014
           PortScan                       90694
           Bot                             1948
Monday     BENIGN                        492605
Thursday   BENIGN                        394718
           Web Attack � Brute Force        1470
           Web Attack � XSS                 652
           Infiltration                      36
           Web Attack � Sql Injection        21
Tuesday    BENIGN                        397317
           FTP

In [2]:
def group(label):
    l = label.strip()
    if l == "BENIGN": return "BENIGN"
    if l in ("FTP-Patator", "SSH-Patator"): return "BruteForce"      # T1110
    if l.startswith("DoS") or l == "Heartbleed": return "DoS"
    if l == "DDoS": return "DDoS"
    if l == "PortScan": return "PortScan"                             # T1046
    if l.startswith("Web Attack"): return "WebAttack"
    if l == "Bot": return "Bot"
    return "Other"                                                    # Infiltration

df["family"] = df["Label"].map(group)
df = df[df["family"] != "Other"]          # drop Infiltration (about 36 rows)
print(df["family"].value_counts())

family
BENIGN        2072254
DoS            193741
DDoS           128014
PortScan        90694
BruteForce       9150
WebAttack        2143
Bot              1948
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, confusion_matrix
from xgboost import XGBClassifier

drop_cols = ["Label", "family", "day", "file"]
X = df.drop(columns=drop_cols)
y = (df["family"] != "BENIGN").astype(int)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
spw = (ytr == 0).sum() / (ytr == 1).sum()
xgb_a = XGBClassifier(n_estimators=200, max_depth=6, scale_pos_weight=spw,
                      n_jobs=-1, random_state=42, tree_method="hist").fit(Xtr, ytr)

def report(model, Xt, yt, name):
    p = model.predict_proba(Xt)[:, 1]
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    print(f"{name}: PR-AUC={average_precision_score(yt, p):.4f} recall={tp/(tp+fn):.4f} "
          f"precision={tp/max(tp+fp,1):.4f} FPR={fp/(fp+tn):.5f}  (FP={fp}, FN={fn})")

report(xgb_a, Xte, yte, "A: random split, all days")

A: random split, all days: PR-AUC=0.9999 recall=0.9995 precision=0.9952 FPR=0.00099  (FP=615, FN=59)


In [4]:
train_days = ["Monday", "Tuesday", "Wednesday", "Thursday"]
tr = df[df["day"].isin(train_days)]
te = df[df["day"] == "Friday"]

Xtr_b, ytr_b = tr.drop(columns=drop_cols), (tr["family"] != "BENIGN").astype(int)
Xte_b, yte_b = te.drop(columns=drop_cols), (te["family"] != "BENIGN").astype(int)

spw_b = (ytr_b == 0).sum() / (ytr_b == 1).sum()
xgb_b = XGBClassifier(n_estimators=200, max_depth=6, scale_pos_weight=spw_b,
                      n_jobs=-1, random_state=42, tree_method="hist").fit(Xtr_b, ytr_b)

report(xgb_b, Xte_b, yte_b, "B: train Mon-Thu, test Fri")

# recall per attack family on Friday
pred_b = (xgb_b.predict_proba(Xte_b)[:, 1] >= 0.5).astype(int)
te = te.assign(pred=pred_b)
print(te[te["family"] != "BENIGN"].groupby("family")["pred"].agg(["mean", "count"])
        .rename(columns={"mean": "recall"}))

B: train Mon-Thu, test Fri: PR-AUC=0.7906 recall=0.0066 precision=0.9613 FPR=0.00015  (FP=59, FN=219189)
            recall   count
family                    
Bot       0.000000    1948
DDoS      0.007984  128014
PortScan  0.004907   90694


In [5]:
cols = [c for c in Xtr_b.columns if c != "Destination Port"]
xgb_b2 = XGBClassifier(n_estimators=200, max_depth=6, scale_pos_weight=spw_b,
                       n_jobs=-1, random_state=42, tree_method="hist").fit(Xtr_b[cols], ytr_b)
report(xgb_b2, Xte_b[cols], yte_b, "B2: no port, train Mon-Thu, test Fri")

B2: no port, train Mon-Thu, test Fri: PR-AUC=0.7390 recall=0.0021 precision=0.8070 FPR=0.00028  (FP=111, FN=220192)


In [6]:
from sklearn.metrics import roc_curve
p = xgb_b.predict_proba(Xte_b)[:, 1]
fpr, tpr, thr = roc_curve(yte_b, p)
for target in [0.001, 0.01, 0.05]:
    i = np.searchsorted(fpr, target, side="right") - 1
    print(f"FPR <= {target}: recall={tpr[i]:.4f} at threshold={thr[i]:.4f}")

FPR <= 0.001: recall=0.3104 at threshold=0.0020
FPR <= 0.01: recall=0.3988 at threshold=0.0001
FPR <= 0.05: recall=0.5039 at threshold=0.0000


In [7]:
import joblib
os.makedirs("../models", exist_ok=True)
joblib.dump(xgb_b, "../models/intrusion_xgb.joblib")
joblib.dump(list(Xtr_b.columns), "../models/intrusion_features.joblib")
df.groupby("day")["family"].value_counts().to_csv("../docs/label_counts.csv")

In [8]:
df["pos"] = df.groupby("file").cumcount() / df.groupby("file")["Label"].transform("count")
is_fri = df["day"] == "Friday"
trC = df[~is_fri | (df["pos"] < 0.5)]
teC = df[is_fri & (df["pos"] >= 0.5)]

dropC = drop_cols + ["pos"]
XtrC, ytrC = trC.drop(columns=dropC), (trC["family"] != "BENIGN").astype(int)
XteC, yteC = teC.drop(columns=dropC), (teC["family"] != "BENIGN").astype(int)

spw_c = (ytrC == 0).sum() / (ytrC == 1).sum()
xgb_c = XGBClassifier(n_estimators=200, max_depth=6, scale_pos_weight=spw_c,
                      n_jobs=-1, random_state=42, tree_method="hist").fit(XtrC, ytrC)
report(xgb_c, XteC, yteC, "C: Mon-Thu + first half of Fri, test second half")

predC = (xgb_c.predict_proba(XteC)[:, 1] >= 0.5).astype(int)
print(teC.assign(pred=predC)[teC["family"] != "BENIGN"]
        .groupby("family")["pred"].agg(["mean", "count"]).rename(columns={"mean": "recall"}))

C: Mon-Thu + first half of Fri, test second half: PR-AUC=0.9986 recall=0.9903 precision=0.9997 FPR=0.00021  (FP=37, FN=1241)
            recall  count
family                   
Bot       0.473138   1638
DDoS      0.999692  58382
PortScan  0.994745  68506


In [11]:
net = pd.DataFrame({"family": teC["family"].values, "label": yteC.values,
                    "score": xgb_c.predict_proba(XteC)[:, 1]})
net.to_csv("../data/processed/network_scores.csv", index=False)

In [9]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_curve

feat = list(Xtr_b.columns)
mon = df[df["day"] == "Monday"].sample(200000, random_state=42)
iso = IsolationForest(n_estimators=200, n_jobs=-1, random_state=42).fit(mon[feat])

score = -iso.score_samples(Xte_b[feat])        # higher = more abnormal
print("IsolationForest PR-AUC:", round(average_precision_score(yte_b, score), 4))

fpr, tpr, _ = roc_curve(yte_b, score)
for target in [0.001, 0.01, 0.05]:
    i = np.searchsorted(fpr, target, side="right") - 1
    print(f"FPR <= {target}: recall={tpr[i]:.4f}")

IsolationForest PR-AUC: 0.456
FPR <= 0.001: recall=0.0000
FPR <= 0.01: recall=0.0000
FPR <= 0.05: recall=0.0976


In [10]:
import joblib, os
os.makedirs("../models", exist_ok=True)
joblib.dump(xgb_b, "../models/intrusion_xgb.joblib")
joblib.dump(feat, "../models/intrusion_features.joblib")

['../models/intrusion_features.joblib']